In [0]:
%run /Workspace/Users/ashishbudz@gmail.com/databricks_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
s3_price = "s3://sportsbar-dp-child-company-prac/gross_price/*.csv"


In [0]:
# WIDGETS
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source","gross_price", "Data Source" )

In [0]:
# WIDGETS values
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
df_bronze = (
    spark.read.format("csv")\
    .option("header", True) \
    .option("inferSchema", True) \
    .load(s3_price)\
    .withColumn("read_timestamp", F.current_timestamp())\
    .select("*", "_metadata.file_name","_metadata.file_size")

)

In [0]:
df_bronze.write.format("delta")\
.option("delta.enableChangeDataFeed", True)\
.mode("overwrite")\
.saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

In [0]:
# Bronze to Silver Layer Processing
df_silver = (
    spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source};')
)

In [0]:
display(df_silver)

In [0]:
# Bringing uniformity in Dates
dates = ['yyyy/MM/dd','dd/MM/yyyy','yyyy-MM-dd','dd-MM-yyyy']

df_silver = df_silver.withColumn(
    "month",
    F.coalesce(
        F.try_to_date(F.col("month"), "yyyy/MM/dd"),
        F.try_to_date(F.col("month"), "dd/MM/yyyy"),
        F.try_to_date(F.col("month"), "yyyy-MM-dd"),
        F.try_to_date(F.col("month"), "dd-MM-yyyy")
    )
)

In [0]:
display(df_silver)

In [0]:
df_silver.printSchema()

In [0]:
# Dealing with negative values in the gross_price column
df_silver = df_silver.withColumn(
    "gross_price",
    F.when(F.col("gross_price").rlike(r'^-?\d+(\.\d+)?$'),
           F.when(F.col("gross_price").cast("double") < 0 , -1 * F.col("gross_price").cast("double"))
           .otherwise(F.col("gross_price").cast("double"))
           ).otherwise(0)
)

In [0]:
display(df_silver)

In [0]:
# Joining with products table to get the Product Code, which is needed when merging with main Gold Table

df_products = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.products;')

In [0]:
display(df_products)

In [0]:
df_products = df_products.select("product_id","product_code")

In [0]:
display(df_products)

In [0]:
df_silver = df_silver.join(df_products, on= "product_id", how = "inner")

In [0]:
display(df_silver)

In [0]:
df_silver = df_silver.select("product_id","product_code","month","gross_price","read_timestamp","file_name","file_size")

In [0]:
display(df_silver)

In [0]:
# Writing to Silver Layer Table
df_silver.write.format("delta")\
    .option("delta.enableChangeDataFeed", True)\
    .option("mergeSchema", True)\
    .mode("overwrite")\
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

In [0]:
#Silver Layer to Gold Layer processing

In [0]:
df_gold = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.{data_source};')

In [0]:
display(df_gold)


In [0]:
# Select only required columns
df_gold = df_gold.select("product_code","month","gross_price")

In [0]:
df_gold.show(5)

In [0]:
# Write to gold table
df_gold.write.format("delta")\
    .option("delta.enableChangeDataFeed", True)\
    .mode("overwrite")\
    .saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{data_source}')

In [0]:
df_gold = (
    df_gold.withColumn(
        "year",
        F.year(F.col("month"))
    ).withColumn(
        "is_zero",
        F.when(F.col("gross_price") == 0, 1).otherwise(0)
    )
)


In [0]:
window = (
    Window.partitionBy("product_code", "year")
    .orderBy(F.col("is_zero"), F.col("month").desc())
)

In [0]:
df_gold = (
    df_gold.withColumn(
        "rnk",
        F.row_number().over(window)
    ).filter(F.col("rnk") == 1)
)

In [0]:
display(df_gold)


In [0]:
# Renaming columns
df_gold = df_gold.withColumnRenamed("gross_price","price_inr")

In [0]:
display(df_gold.show(5))

In [0]:
# Selecting only necessary columns
df_gold = df_gold.select("product_code","price_inr","year")

# Changing year to String
df_gold = df_gold.withColumn("year", F.col("year").cast("string"))

In [0]:
target = DeltaTable.forName(spark,f'{catalog}.{gold_schema}.dim_{data_source}')

target.alias("target").merge(
    source = df_gold.alias("source"),
    condition = "target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "price_inr" : "source.price_inr",
        "year" : "source.year"
    }
).whenNotMatchedInsert(
    values={
        "product_code" : "source.product_code",
        "price_inr" : "source.price_inr",
        "year" : "source.year"
    }
).execute()